# 산술 갈래 진단 — 로컬 리더, Colab GPU

**묻는 것 하나.** 로컬 Qwen2.5-7B가 산술 질문에서 arm 무관 `.02~.06`인 원인이
**4비트 압축**인가, **7B라는 크기**인가?

같은 모델·같은 검색·같은 채점으로 **압축만** 바꿔 두 번 돌리고 비교한다.
압축이 범인이면 → 8비트로 본표 전체를 다시 돌릴 근거가 생긴다.
크기가 범인이면 → T4로는 안 풀리고 14B 이상이 필요하다.

> Colab GPU는 **API가 아니다.** 가중치를 직접 받아 이 프로세스에서 돌리므로
> `local:` 스펙 그대로고, "리더는 로컬" 제약을 깨지 않는다. 빌린 것은 GPU뿐이다.

**셀을 위에서부터 순서대로 실행.** 2번은 한 번만 하면 되고, 세션이 끊기면
1·2·3을 다시 돌린 뒤 5번부터 이어가면 된다 (resume 내장).


## 1. GPU 확인 — 무엇을 돌릴 수 있는지 여기서 정해진다

런타임 → 런타임 유형 변경 → **T4 GPU** 선택 후 실행.


In [ ]:
import subprocess, torch
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total',
                      '--format=csv,noheader'], capture_output=True, text=True).stdout.strip())
assert torch.cuda.is_available(), 'GPU 런타임이 아닙니다: 런타임 유형 변경 → T4 GPU'
GB = torch.cuda.get_device_properties(0).total_memory / 1024**3
CAP = torch.cuda.get_device_capability(0)
print(f'VRAM {GB:.1f} GB | compute capability {CAP[0]}.{CAP[1]}')

# Turing(7.5, T4)에는 bfloat16 하드웨어가 없다 -- 돌기는 하지만 느리다
DTYPE = 'bfloat16' if CAP[0] >= 8 else 'float16'
# 7B 가중치: 4bit 약 5GB / 8bit 약 8GB / 무압축 약 15GB(+KV 캐시)
FULL_OK = GB >= 20
print(f'dtype={DTYPE} | 무압축 비교 가능: {FULL_OK}')
print('무압축이 불가하면 4bit vs 8bit로 비교한다 -- 묻는 것은 같다.')


## 2. 저장소 + 의존성 (세션당 한 번)

비공개 저장소면 `TOKEN`에 GitHub PAT를 넣는다 (Settings → Developer settings →
Personal access tokens → repo 읽기 권한). 공개면 빈 문자열로 두면 된다.


In [ ]:
TOKEN = ''   # 비공개 저장소면 GitHub PAT, 공개면 빈 칸
BRANCH = 'fix/encoder-provenance'

import os, pathlib
url = f'https://{TOKEN}@github.com/Jax0303/T2.git' if TOKEN else 'https://github.com/Jax0303/T2.git'
if not pathlib.Path('/content/T2').exists():
    !git clone --depth 1 -b {BRANCH} {url} /content/T2
os.chdir('/content/T2/rag-agent')
!git log --oneline -1

# torch는 Colab에 이미 있다. 나머지만.
!pip -q install 'transformers>=4.40' 'bitsandbytes>=0.43' 'accelerate' \
                'sentence-transformers>=3.0' 'faiss-cpu>=1.7' 'rank-bm25==0.2.2' \
                'scipy>=1.11' 'scikit-learn>=1.4' 'tabulate>=0.9' 2>&1 | tail -2
print('done')


## 3. HiTab 데이터 (세션당 한 번, 약 94MB)

데이터는 저장소에 없다 (`.gitignore`). 공식 릴리스에서 받는다.


In [ ]:
!python scripts/download_hitab.py --dest data/hitab 2>&1 | tail -3
!ls data/hitab/data/ && wc -l populations/hitab_dev_corpus_arith.txt


## 4. Drive에 결과를 내보낼 준비 (선택, 권장)

Colab 세션이 죽으면 `/content`는 사라진다. Drive에 두면 다음 세션에서
**중단 지점부터 이어서** 돌릴 수 있다 (`--force-resume` 불필요, 자동).


In [ ]:
USE_DRIVE = True

import pathlib, shutil
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE = pathlib.Path('/content/drive/MyDrive/T2_arith')
    SAVE.mkdir(parents=True, exist_ok=True)
    # 지난 세션 결과가 있으면 되살린다 -> resume이 이를 읽는다
    for f in SAVE.glob('*'):
        shutil.copy(f, pathlib.Path('results') / f.name)
    print('복원:', [f.name for f in SAVE.glob('*')] or '(없음)')
else:
    SAVE = None


## 5. 기준 조건 — 지금 논문이 쓰는 리더 (4비트)

이미 측정된 `.02~.06`을 이 환경에서 재현하는 셀이다. **비교의 기준선**이므로
건너뛰지 말 것 — GPU가 다르면 절대값이 조금 움직일 수 있고, 두 조건이 같은
기기에서 나와야 차이를 압축 탓으로 돌릴 수 있다.

약 178문항 × 4 arm. T4에서 대략 40~70분.


In [ ]:
BUDGET = 512
ARMS   = 'dump,cell,row,flat'

!python scripts/corpus_dump_vs_cell.py \
  --dataset hitab --population hitab_dev_corpus_arith \
  --retriever dense --budget {BUDGET} --cell-scheme S3 --arms {ARMS} \
  --reader "local:Qwen/Qwen2.5-7B-Instruct?quantization=4bit&dtype={DTYPE}" \
  --out results/arith_local_4bit_{BUDGET}.json 2>&1 | tail -20


In [ ]:
# 세션이 끊길 수 있으니 끝나는 대로 Drive에 복사
if SAVE:
    for f in pathlib.Path('results').glob(f'arith_local_*_{BUDGET}*'):
        shutil.copy(f, SAVE / f.name)
    print('저장:', [f.name for f in SAVE.glob('*')])


## 6. 대조 조건 — 압축을 완화한 같은 모델

**바꾸는 것은 압축 하나뿐이다.** 모델·검색·예산·프롬프트·채점기 전부 동일.
`quantization`이 리더 이름에 들어가므로, 5번 결과 위에 잘못 이어붙는 일은 없다
(`guard_resume`이 거부한다).


In [ ]:
QUANT = 'none' if FULL_OK else '8bit'
print(f'대조 조건: quantization={QUANT}')

!python scripts/corpus_dump_vs_cell.py \
  --dataset hitab --population hitab_dev_corpus_arith \
  --retriever dense --budget {BUDGET} --cell-scheme S3 --arms {ARMS} \
  --reader "local:Qwen/Qwen2.5-7B-Instruct?quantization={QUANT}&dtype={DTYPE}" \
  --out results/arith_local_{QUANT}_{BUDGET}.json 2>&1 | tail -20


In [ ]:
if SAVE:
    for f in pathlib.Path('results').glob(f'arith_local_*_{BUDGET}*'):
        shutil.copy(f, SAVE / f.name)
    print('저장:', [f.name for f in SAVE.glob('*')])


## 7. 판정

arm별 EM을 나란히 놓고, 같은 질문끼리 짝지어 검정한다.


In [ ]:
import json, itertools
from scipy.stats import binomtest

def load(tag):
    p = f'results/arith_local_{tag}_{BUDGET}_records.jsonl'
    return {r['query_id']: r for r in map(json.loads, open(p))}

A, B = load('4bit'), load(QUANT)
both = sorted(set(A) & set(B))
print(f'짝지어진 질문 {len(both)}개  |  4bit  vs  {QUANT}\n')
print(f"{'arm':<8}{'4bit':>8}{QUANT:>10}{'차이':>8}{'승:패':>10}{'p':>10}")
for arm in ARMS.split(','):
    a = [A[q][arm]['answer_em'] for q in both]
    b = [B[q][arm]['answer_em'] for q in both]
    win  = sum(1 for x, y in zip(a, b) if y > x)   # 대조 조건만 맞힌 것
    lose = sum(1 for x, y in zip(a, b) if y < x)
    p = binomtest(win, win + lose).pvalue if win + lose else 1.0
    ma, mb = sum(a)/len(a), sum(b)/len(b)
    print(f'{arm:<8}{ma:>8.3f}{mb:>10.3f}{mb-ma:>+8.3f}{f"{win}:{lose}":>10}{p:>10.4f}')

print()
cell_gain = (sum(B[q]['cell']['answer_em'] for q in both)
             - sum(A[q]['cell']['answer_em'] for q in both)) / len(both)
if cell_gain >= 0.05:
    print('판정: 압축이 범인이다. 본표 전체를 이 설정으로 다시 돌릴 근거가 생겼다.')
elif cell_gain <= 0.01:
    print('판정: 압축이 아니다 -- 7B라는 크기의 문제.')
    print('      T4로는 안 풀린다. 14B 이상이 필요하고, 그것은 별도 결정이다.')
else:
    print('판정: 애매하다. 예산 256에서도 같은 부호가 나오는지 확인할 것.')


## 다음

| 결과 | 다음 할 일 |
|---|---|
| 압축이 범인 | 본표(HiTab·MultiHiertt·AIT-QA) 전체를 8비트로 재실행. 리더가 바뀌므로 **모든 arm·모든 데이터셋**을 다시 돌려야 비교가 성립한다 |
| 크기가 범인 | Qwen2.5-14B(4비트, 약 9GB)로 같은 진단. T4에서 아슬아슬하므로 A100/L4 런타임 권장 |
| 애매 | 예산 256에서 반복 |

어느 쪽이든 **MultiHiertt가 다음 차례다** — 그 모집단은 절반(200/400)이 집계형이라,
리더의 산술 실패가 이미 본표 숫자를 오염시키고 있다.
